# Sel 1: gabungkan semua tabel

In [1]:
import pandas as pd, numpy as np
from pathlib import Path
D = Path('/content/data')

def read(name, cols, skip):
    d = pd.read_csv(D / name, encoding='utf-8-sig', header=None, skiprows=skip, dtype=str).iloc[:, :len(cols)]
    d.columns = cols
    d['prov'] = d['prov'].str.strip()
    for c in cols[1:]: d[c] = pd.to_numeric(d[c], errors='coerce')
    return d

ahh = read('Angka_Harapan_Hidup__AHH__Menurut_Provinsi_dan_Jenis_Kelamin__2024.csv', ['prov', 'ahh_l', 'ahh_p'], 4)
pov = read('Persentase_Penduduk_Miskin__P0__Menurut_Provinsi_dan_Daerah__2024.csv',
           ['prov', 'u1', 'u2', 'u3', 'r1', 'r2', 'r3', 'pov_mar', 'pov_sep', 'pov_yr'], 5)[['prov', 'pov_mar', 'pov_sep']]
gdp = read('_Seri_2010__Produk_Domestik_Regional_Bruto_Per_Kapita__2024.csv', ['prov', 'gdp_cur', 'gdp_const'], 4)
aps = read('Angka_Partisipasi_Sekolah__APS__Menurut_Provinsi_dan_Kelompok_Umur__2024.csv',
           ['prov', 'aps_7_12', 'aps_13_15', 'aps_16_18', 'aps_19_23'], 4)
san = read('Proporsi_Rumah_Tangga_Yang_Memiliki_Akses_Terhadap_Layanan_Sanitasi_Layak__2024.csv', ['prov', 'sanitasi'], 3)

df = ahh
for t in (pov, gdp, aps, san): df = df.merge(t, on='prov', how='outer', validate='1:1')
national = df[df.prov == 'INDONESIA'].copy()
df = df[df.prov != 'INDONESIA'].copy()

df['ahh'] = (df.ahh_l + df.ahh_p) / 2            # rata-rata L dan P, bukan AHH total resmi BPS
df['ahh_gap'] = df.ahh_p - df.ahh_l
df['pov'] = (df.pov_mar + df.pov_sep) / 2
df['lgdp'] = np.log(df.gdp_cur)

REG = {'Sumatera': ['ACEH', 'SUMATERA UTARA', 'SUMATERA BARAT', 'RIAU', 'JAMBI', 'SUMATERA SELATAN', 'BENGKULU', 'LAMPUNG', 'KEP. BANGKA BELITUNG', 'KEP. RIAU'],
       'Jawa': ['DKI JAKARTA', 'JAWA BARAT', 'JAWA TENGAH', 'DI YOGYAKARTA', 'JAWA TIMUR', 'BANTEN'],
       'Bali-Nusa Tenggara': ['BALI', 'NUSA TENGGARA BARAT', 'NUSA TENGGARA TIMUR'],
       'Kalimantan': ['KALIMANTAN BARAT', 'KALIMANTAN TENGAH', 'KALIMANTAN SELATAN', 'KALIMANTAN TIMUR', 'KALIMANTAN UTARA'],
       'Sulawesi': ['SULAWESI UTARA', 'SULAWESI TENGAH', 'SULAWESI SELATAN', 'SULAWESI TENGGARA', 'GORONTALO', 'SULAWESI BARAT'],
       'Maluku': ['MALUKU', 'MALUKU UTARA'],
       'Papua': ['PAPUA BARAT', 'PAPUA BARAT DAYA', 'PAPUA', 'PAPUA SELATAN', 'PAPUA TENGAH', 'PAPUA PEGUNUNGAN']}
m = {p: r for r, ps in REG.items() for p in ps}
df['wilayah'] = df.prov.map(m)
df['papua'] = df.wilayah.eq('Papua')

use = ['ahh', 'ahh_l', 'ahh_p', 'pov', 'gdp_cur', 'lgdp', 'aps_7_12', 'aps_13_15', 'aps_16_18', 'aps_19_23', 'sanitasi']
assert len(df) == 38 and df.prov.is_unique
assert df.wilayah.notna().all(), df[df.wilayah.isna()].prov.tolist()
assert df[use].notna().all().all()
print(df.wilayah.value_counts().to_dict())
print(df[use].describe().round(2).T[['min', '50%', 'max']])

out = D / 'province_health_2024.csv'; df.to_csv(out, index=False, encoding='utf-8')
print('tersimpan:', out, df.shape)

{'Sumatera': 10, 'Jawa': 6, 'Sulawesi': 6, 'Papua': 6, 'Kalimantan': 5, 'Bali-Nusa Tenggara': 3, 'Maluku': 2}
                min       50%        max
ahh           64.76     70.73      75.53
ahh_l         62.83     68.76      73.66
ahh_p         66.68     72.70      77.40
pov            3.90      9.85      31.32
gdp_cur    18105.38  68596.08  344382.53
lgdp           9.80     11.14      12.75
aps_7_12      74.63     99.38      99.71
aps_13_15     68.42     96.16      98.91
aps_16_18     47.65     75.53      90.36
aps_19_23     17.84     30.30      54.17
sanitasi      12.61     83.36      96.83
tersimpan: /content/data/province_health_2024.csv (38, 18)


# Sel 2: kesenjangan (pertanyaan 1 dan 5, deskriptif)

In [2]:
import pandas as pd, numpy as np
df = pd.read_csv('/content/data/province_health_2024.csv')

def spread(s):
    q1, q3 = s.quantile([.25, .75])
    return pd.Series({'min': s.min(), 'median': s.median(), 'max': s.max(),
                      'range': s.max() - s.min(), 'IQR': q3 - q1, 'SD': s.std()})
print(df[['ahh', 'ahh_l', 'ahh_p']].apply(spread).round(2).T)

for c in ['ahh', 'ahh_l', 'ahh_p']:
    print(f'\n{c}: 5 tertinggi'); print(df.nlargest(5, c)[['prov', 'wilayah', c]].round(2).to_string(index=False))
    print(f'{c}: 5 terendah');  print(df.nsmallest(5, c)[['prov', 'wilayah', c]].round(2).to_string(index=False))

print('\nPer wilayah (AHH rata-rata L dan P):')
print(df.groupby('wilayah').ahh.agg(['count', 'median', 'min', 'max']).round(2).sort_values('median', ascending=False))

print('\nSelisih perempuan - laki-laki (tahun):'); print(df.ahh_gap.describe().round(2))
print(df.groupby('wilayah').ahh_gap.agg(['median', 'min', 'max']).round(2))
print(df.nlargest(3, 'ahh_gap')[['prov', 'ahh_gap']].round(2).to_string(index=False))
print(df.nsmallest(3, 'ahh_gap')[['prov', 'ahh_gap']].round(2).to_string(index=False))

         min  median    max  range   IQR    SD
ahh    64.76   70.73  75.53  10.78  3.45  2.66
ahh_l  62.83   68.76  73.66  10.83  3.45  2.65
ahh_p  66.68   72.70  77.40  10.72  3.45  2.67

ahh: 5 tertinggi
            prov    wilayah   ahh
   DI YOGYAKARTA       Jawa 75.53
     JAWA TENGAH       Jawa 75.10
KALIMANTAN TIMUR Kalimantan 75.06
      JAWA BARAT       Jawa 74.41
     DKI JAKARTA       Jawa 74.18
ahh: 5 terendah
            prov  wilayah   ahh
PAPUA PEGUNUNGAN    Papua 64.76
  SULAWESI BARAT Sulawesi 66.32
   PAPUA SELATAN    Papua 66.47
          MALUKU   Maluku 67.00
    PAPUA TENGAH    Papua 67.01

ahh_l: 5 tertinggi
            prov    wilayah  ahh_l
   DI YOGYAKARTA       Jawa  73.66
     JAWA TENGAH       Jawa  73.21
KALIMANTAN TIMUR Kalimantan  73.21
      JAWA BARAT       Jawa  72.26
     DKI JAKARTA       Jawa  72.08
ahh_l: 5 terendah
            prov  wilayah  ahh_l
PAPUA PEGUNUNGAN    Papua  62.83
  SULAWESI BARAT Sulawesi  64.37
   PAPUA SELATAN    Papua  64.53
  

# Sel 3: korelasi dengan interval bootstrap (pertanyaan 2 dan semua uji kepekaan)

In [3]:
from scipy.stats import spearmanr
rng = np.random.default_rng(42)       # seed dicatat di README
B = 2000

def boot_rho(d, x, y='ahh'):
    r = spearmanr(d[x], d[y]).statistic
    n, bs = len(d), []
    for _ in range(B):
        s = d.iloc[rng.integers(0, n, n)]
        if s[x].nunique() < 2 or s[y].nunique() < 2: continue
        bs.append(spearmanr(s[x], s[y]).statistic)
    lo, hi = np.percentile(bs, [2.5, 97.5]); return r, lo, hi

factors = ['pov', 'sanitasi', 'aps_16_18', 'lgdp']
def table(d, y='ahh'):
    return pd.DataFrame({f: boot_rho(d, f, y) for f in factors}, index=['rho', 'lo95', 'hi95']).T.round(2)

print('SEMUA PROVINSI (n=38)'); print(table(df))
nop = df[~df.papua]
print(f'\nTANPA PAPUA (n={len(nop)})'); print(table(nop))
print('\nLAKI-LAKI'); print(table(df, 'ahh_l'))
print('\nPEREMPUAN'); print(table(df, 'ahh_p'))
print('\nLAMPIRAN: semua kelompok APS (n=38)')
print(pd.DataFrame({c: boot_rho(df, c) for c in ['aps_7_12', 'aps_13_15', 'aps_16_18', 'aps_19_23']},
                   index=['rho', 'lo95', 'hi95']).T.round(2))

SEMUA PROVINSI (n=38)
            rho  lo95  hi95
pov       -0.65 -0.81 -0.38
sanitasi   0.66  0.38  0.83
aps_16_18  0.21 -0.16  0.52
lgdp       0.33 -0.04  0.65

TANPA PAPUA (n=32)
            rho  lo95  hi95
pov       -0.44 -0.71 -0.07
sanitasi   0.53  0.21  0.77
aps_16_18  0.11 -0.27  0.47
lgdp       0.43  0.02  0.73

LAKI-LAKI
            rho  lo95  hi95
pov       -0.65 -0.81 -0.36
sanitasi   0.66  0.40  0.84
aps_16_18  0.21 -0.15  0.53
lgdp       0.33 -0.07  0.63

PEREMPUAN
            rho  lo95  hi95
pov       -0.65 -0.81 -0.37
sanitasi   0.66  0.37  0.84
aps_16_18  0.21 -0.16  0.53
lgdp       0.33 -0.05  0.64

LAMPIRAN: semua kelompok APS (n=38)
            rho  lo95  hi95
aps_7_12   0.59  0.31  0.79
aps_13_15  0.46  0.11  0.72
aps_16_18  0.21 -0.15  0.52
aps_19_23  0.02 -0.35  0.35


# Sel 4: regresi, VIF, residual, dan uji kepekaan

In [4]:
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
P3 = ['pov', 'sanitasi', 'aps_16_18']

def ols_boot(d, preds, B=2000):
    Z = d[preds].apply(lambda s: (s - s.mean()) / s.std()).values          # prediktor distandardkan
    X = np.column_stack([np.ones(len(d)), Z]); y = d['ahh'].values
    beta = np.linalg.lstsq(X, y, rcond=None)[0]
    res = y - X @ beta
    r2 = 1 - res.var() / y.var(); n, p = len(y), len(preds)
    adj = 1 - (1 - r2) * (n - 1) / (n - p - 1)
    bs = [np.linalg.lstsq(X[i], y[i], rcond=None)[0] for i in (rng.integers(0, n, n) for _ in range(B))]
    lo, hi = np.percentile(bs, [2.5, 97.5], axis=0)
    out = pd.DataFrame({'coef': beta, 'lo95': lo, 'hi95': hi}, index=['const'] + preds).round(2)
    out.loc[preds, 'VIF'] = np.round([variance_inflation_factor(X, j) for j in range(1, X.shape[1])], 2)
    return out, round(r2, 3), round(adj, 3), res

for name, d, preds in [('UTAMA: 3 prediktor, semua provinsi', df, P3),
                       ('TANPA PAPUA: 3 prediktor', df[~df.papua], P3),
                       ('KEPEKAAN: 4 prediktor (+ log PDRB)', df, P3 + ['lgdp'])]:
    out, r2, adj, res = ols_boot(d, preds)
    print(f'\n{name} | n={len(d)} | R2={r2} | R2 adj={adj}'); print(out)
    if name.startswith('UTAMA'): df['resid'] = res

print('\nResidual positif terbesar (AHH lebih tinggi dari prediksi):')
print(df.nlargest(5, 'resid')[['prov', 'wilayah', 'ahh', 'resid']].round(2).to_string(index=False))
print('Residual negatif terbesar:')
print(df.nsmallest(5, 'resid')[['prov', 'wilayah', 'ahh', 'resid']].round(2).to_string(index=False))

print('\nPertanyaan 5, eksploratif: korelasi selisih P-L dengan faktor')
print(pd.DataFrame({f: boot_rho(df, f, 'ahh_gap') for f in factors}, index=['rho', 'lo95', 'hi95']).T.round(2))
df.to_csv('/content/data/province_health_2024_resid.csv', index=False)


UTAMA: 3 prediktor, semua provinsi | n=38 | R2=0.52 | R2 adj=0.478
            coef   lo95   hi95   VIF
const      70.58  69.85  71.14   NaN
pov        -1.47  -2.48  -0.52  2.54
sanitasi    0.57  -0.51   2.50  3.76
aps_16_18  -0.03  -0.88   1.11  2.02

TANPA PAPUA: 3 prediktor | n=32 | R2=0.365 | R2 adj=0.298
            coef   lo95   hi95   VIF
const      71.25  70.61  71.92   NaN
pov        -0.87  -1.69  -0.16  1.14
sanitasi    0.72  -0.22   1.61  1.32
aps_16_18   0.25  -0.55   1.00  1.18

KEPEKAAN: 4 prediktor (+ log PDRB) | n=38 | R2=0.546 | R2 adj=0.49
            coef   lo95   hi95   VIF
const      70.58  69.87  71.13   NaN
pov        -1.31  -2.29   0.03  2.68
sanitasi    0.50  -0.62   2.88  3.78
aps_16_18   0.01  -0.94   1.08  2.02
lgdp        0.46  -0.25   1.13  1.20

Residual positif terbesar (AHH lebih tinggi dari prediksi):
            prov    wilayah   ahh  resid
   DI YOGYAKARTA       Jawa 75.53   4.36
     JAWA TENGAH       Jawa 75.10   4.10
      JAWA BARAT       Jawa 7

# Sel 5: simpan hasil final untuk dashboard

In [6]:
from pathlib import Path
OUT = Path('/content/data/processed'); OUT.mkdir(exist_ok=True, parents=True)
rng = np.random.default_rng(42)

a = table(df); a['scenario'] = 'All provinces (n=38)'
b = table(nop); b['scenario'] = f'Without Papua (n={len(nop)})'
pd.concat([a, b]).reset_index().rename(columns={'index': 'factor'}).to_csv(OUT / 'corr.csv', index=False)

rows = []
for name, d, preds in [('Main model (n=38)', df, P3), (f'Without Papua (n={len(nop)})', nop, P3),
                       ('With log GRDP (n=38)', df, P3 + ['lgdp'])]:
    o, r2, adj, res = ols_boot(d, preds)
    o = o.reset_index().rename(columns={'index': 'term'}); o['scenario'] = name; o['r2'] = r2; o['r2_adj'] = adj
    rows.append(o)
    if name.startswith('Main'): df['resid'] = res
pd.concat(rows).to_csv(OUT / 'model.csv', index=False)
df.to_csv(OUT / 'province_health_2024.csv', index=False)
print(pd.concat(rows).round(2).to_string(index=False)); print(pd.read_csv(OUT / 'corr.csv').round(2).to_string(index=False))

     term  coef  lo95  hi95  VIF             scenario   r2  r2_adj
    const 70.58 69.78 71.14  NaN    Main model (n=38) 0.52    0.48
      pov -1.47 -2.50 -0.53 2.54    Main model (n=38) 0.52    0.48
 sanitasi  0.57 -0.66  2.47 3.76    Main model (n=38) 0.52    0.48
aps_16_18 -0.03 -0.90  1.17 2.02    Main model (n=38) 0.52    0.48
    const 71.25 70.57 71.92  NaN Without Papua (n=32) 0.36    0.30
      pov -0.87 -1.63 -0.17 1.14 Without Papua (n=32) 0.36    0.30
 sanitasi  0.72 -0.12  1.64 1.32 Without Papua (n=32) 0.36    0.30
aps_16_18  0.25 -0.59  1.01 1.18 Without Papua (n=32) 0.36    0.30
    const 70.58 69.84 71.17  NaN With log GRDP (n=38) 0.55    0.49
      pov -1.31 -2.30  0.01 2.68 With log GRDP (n=38) 0.55    0.49
 sanitasi  0.50 -0.60  2.82 3.78 With log GRDP (n=38) 0.55    0.49
aps_16_18  0.01 -1.02  1.08 2.02 With log GRDP (n=38) 0.55    0.49
     lgdp  0.46 -0.23  1.19 1.20 With log GRDP (n=38) 0.55    0.49
   factor   rho  lo95  hi95             scenario
      pov -0.